# Event Power Rating (EPR) Tracking
This notebook reads the ORWIL EPA data, steps through each match, and keeps a running EPR for every team based on the opposing alliance net EPA.

In [2]:
from pathlib import Path
import json
import pandas as pd

data_path = Path('betterSB/Cais_epa_data.json')
updates_path = Path('Cais_EPR_updates.json')
csv_path = Path('CaisEPR.csv')

with data_path.open() as fh:
    matches = json.load(fh)['matches']

def adaptive_margin(match_index: int) -> float:
    if match_index <= 6:
        return 0.5
    if match_index >= 12:
        return 0.3
    return 0.5 - (match_index - 6) * (0.2 / 6)

team_epr: dict[int, float] = {}
match_records = []

for match_index, match in enumerate(matches, start=1):
    alliance_groups = match['teams']
    red_teams = [team for team in alliance_groups if team['alliance'] == 'red']
    blue_teams = [team for team in alliance_groups if team['alliance'] == 'blue']
    red_net = sum(team['epa']['epa_change'] for team in red_teams)
    blue_net = sum(team['epa']['epa_change'] for team in blue_teams)
    margin = adaptive_margin(match_index)
    blue_delta = -(red_net / 3.0) * margin
    red_delta = -(blue_net / 3.0) * margin

    match_info = {
        'match_number': match_index,
        'match_key': match.get('match_key'),
        'red_net_epa': red_net,
        'blue_net_epa': blue_net,
        'margin': margin,
        'teams': [],
    }

    for team in blue_teams:
        team_epr.setdefault(team['team'], 0.0)
        team_epr[team['team']] += blue_delta
        match_info['teams'].append({
            'team': team['team'],
            'alliance': 'blue',
            'epr_change': blue_delta,
        })

    for team in red_teams:
        team_epr.setdefault(team['team'], 0.0)
        team_epr[team['team']] += red_delta
        match_info['teams'].append({
            'team': team['team'],
            'alliance': 'red',
            'epr_change': red_delta,
        })

    match_records.append(match_info)

updates_path.write_text(json.dumps({'matches': match_records}, indent=2))

epr_df = (
    pd.DataFrame.from_dict(team_epr, orient='index', columns=['epr'])
    .reset_index()
    .rename(columns={'index': 'team'})
    .sort_values('team')
)
epr_df.to_csv(csv_path, index=False, float_format='%.6f')

print('wrote', updates_path, 'and', csv_path)
print(epr_df.head(3).to_string(index=False))


TypeError: unsupported operand type(s) for +: 'int' and 'NoneType'